In [1]:
import os
import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from igraph import Graph, plot
import matplotlib.pyplot as plt

# SAVE = False  # whether to save the outputs of this notebook

# if SAVE:
#     # Replace this with the path to directory where you would like results to be saved
#     OUTPUT_PATH = "../output_files/synthetic_example"
#     os.makedirs(OUTPUT_PATH, exist_ok=True)  # Create path if it doesn't exist

In [2]:
sc.settings.verbosity = 3

In [3]:
adata = sc.read_h5ad("/home/jovyan/work/results/discrete_cancer_stages_annotated.h5ad")
adata

AnnData object with n_obs × n_vars = 122649 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'cell_type', 'cell_cycle', 'Gender', 'Age', 'Shimada', 'INSS_stage', 'COG_risk_group', 'Distant_metastasis', 'NSE', 'LDH', 'MYCN-amp', 'Survival/Death', 'discrete_cancer_stage'
    var: 'mt', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'INSS_stage_colors', 'hvg', 'leiden', 'leiden_colors', 'leiden_sizes', 'log1p', 'neighbors', 'paga', 'pca', 'sample_ID_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    obsp: 'connectivities', 'distances'

In [4]:
rdata = sc.read_h5ad("/home/jovyan/work/results/raw.h5ad")
rdata

AnnData object with n_obs × n_vars = 197850 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [5]:
rdata = rdata[adata.obs_names]
rdata

View of AnnData object with n_obs × n_vars = 122649 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [6]:
rdata.obs["discrete_cancer_stage"] = adata.obs["discrete_cancer_stage"]
rdata

/tmp/ipykernel_28311/848089521.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  rdata.obs["discrete_cancer_stage"] = adata.obs["discrete_cancer_stage"]


AnnData object with n_obs × n_vars = 122649 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [7]:
# floats
rdata.X

array([[ 0. ,  0. ,  0. , ...,  0. ,  7. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  3.5,  0. ],
       [ 0. ,  0. ,  1. , ...,  0. ,  4. ,  0. ],
       ...,
       [ 0. ,  0. ,  0. , ...,  0. ,  0.5,  0. ],
       [ 0. ,  4. ,  0. , ...,  0. , 11. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. , 16. ,  0. ]], dtype=float32)

In [8]:
# input should be only integers
rdata.X = np.round(rdata.X)
# Convert to integers, rounding up if decimal >= 0.5
rdata.X = rdata.X.astype(int)
rdata.X

array([[ 0,  0,  0, ...,  0,  7,  0],
       [ 0,  0,  0, ...,  0,  4,  0],
       [ 0,  0,  1, ...,  0,  4,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  4,  0, ...,  0, 11,  0],
       [ 0,  0,  0, ...,  0, 16,  0]])

In [9]:
metadata = pd.DataFrame(rdata.obs["discrete_cancer_stage"], columns=["discrete_cancer_stage"])
metadata

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGAGGTTGC.1,I
T44_AAACCTGAGAGTGAGA.1,4S
T44_AAACCTGAGGCATGTG.1,IV
T44_AAACCTGAGTGTACCT.1,4S
T44_AAACCTGCAATGCCAT.1,IV
...,...
T40_TTTGGTTAGCAATATG.1,I
T40_TTTGGTTAGTGGAGTC.1,IV
T40_TTTGGTTCATAGTAAG.1,IV


4S_I

In [10]:
counts_4S_I = rdata[rdata.obs.discrete_cancer_stage.isin(["4S", "I"])]
print(counts_4S_I)
counts_4S_I_df = pd.DataFrame(counts_4S_I.X, index=counts_4S_I.obs_names, columns=counts_4S_I.var_names)
counts_4S_I_df

View of AnnData object with n_obs × n_vars = 39665 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,LAS1L,ENPP4,ANKIB1,CYP51A1,KRIT1,RAD52,BAD,LAP3,CD99,HS3ST1,HECW1,MAD1L1,LASP1,SNX11,TMEM176A,M6PR,KLHL13,ICA1,DBNDD1,ALS2,CASP10,CFLAR,NDUFAF7,RBM5,MTMR7,ARF5,SARM1,POLDIP2,PLXND1,AK2,...,DHRS11,ACACA,TMEM191B,C11orf98,MRM1,GOLGA6L10,HIST1H4B,NKILA,BACE1-AS,AC007325.4,HIST1H3H,MRPL45,AL358852.1,AP006285.2,SND1-IT1,AC008522.1,AC009065.4,UCKL1-AS1,AF131216.1,AL365181.2,AC245100.1,AC133644.2,PAGR1,LINC00294,CTBP1-AS,TRG-AS1,LINC00997,ARRDC3-AS1,RASSF1-AS1,LINC01176,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,RGS5,MATR3
Cell_barcode,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGGTTGC.1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,1,0,1,0,1,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,0
T44_AAACCTGAGAGTGAGA.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGAGTGTACCT.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,0
T44_AAACCTGCAATGGATA.1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
T44_AAACCTGCACTGTGTA.1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTCTTAGTCTCTAGGA.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,2,0,0,0,1,0,0,0,3,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,0
T40_TTGTAGGGTCTCTTTA.1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,5,0,0,0,0,0,0,0,3,0,0,0,1,...,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
T40_TTTCCTCAGATGCCAG.1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,3,0,0,0,0,0,0,0,2,0,0,1,0,0,0,0,0,0,0,3,0,1,0,0,...,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
metadata_4S_I = metadata[metadata.discrete_cancer_stage.isin(["4S", "I"])]
metadata_4S_I

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGAGGTTGC.1,I
T44_AAACCTGAGAGTGAGA.1,4S
T44_AAACCTGAGTGTACCT.1,4S
T44_AAACCTGCAATGGATA.1,4S
T44_AAACCTGCACTGTGTA.1,4S
...,...
T40_TTCTTAGTCTCTAGGA.1,4S
T40_TTGTAGGGTCTCTTTA.1,I
T40_TTTCCTCAGATGCCAG.1,4S


In [12]:
inference = DefaultInference(n_cpus=None)

In [13]:
# generate object
dds_4S_I = DeseqDataSet(
    counts=counts_4S_I_df,
    metadata=metadata_4S_I,
    design_factors="discrete_cancer_stage",
    refit_cooks=True,
    inference=inference,
    # n_cpus=8, # n_cpus can be specified here or in the inference object
)
dds_4S_I

/tmp/ipykernel_28311/2813601622.py:2: UserWarning: Same factor names in the design contain underscores ('_'). They will
                be converted to hyphens ('-').
  dds_4S_I = DeseqDataSet(


AnnData object with n_obs × n_vars = 39665 × 12850
    obs: 'discrete-cancer-stage'
    obsm: 'design_matrix'

In [ ]:
dds_4S_I.deseq2()

Fitting size factors...
/opt/conda/envs/the-project/lib/python3.10/site-packages/pydeseq2/dds.py:441: UserWarning: Every gene contains at least one zero, cannot compute log geometric means. Switching to iterative mode.
  self.fit_size_factors()
Fitting dispersions...
... done in 36.53 seconds.

Fitting MAP dispersions...
... done in 124.77 seconds.



In [ ]:
# save the file
with open("dds_4S_I.pkl", "wb") as f:
        pkl.dump(dds_4S_I, f)

4S_III

In [15]:
counts_4S_III = rdata[rdata.obs.discrete_cancer_stage.isin(["4S", "III"])]
print(counts_4S_III)
counts_4S_III_df = pd.DataFrame(counts_4S_III.X, index=counts_4S_III.obs_names, columns=counts_4S_III.var_names)
counts_4S_III_df

View of AnnData object with n_obs × n_vars = 43715 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,RGS5,MATR3
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGTGAGA.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGAGTGTACCT.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,0
T44_AAACCTGCAATGGATA.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
T44_AAACCTGCACTGTGTA.1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGGTAATCACC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTCTTAGCAGTCTTCC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T40_TTCTTAGTCTCTAGGA.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,0
T40_TTGTAGGCAGCGAACA.1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,5,0


In [16]:
counts_4S_IV = rdata[rdata.obs.discrete_cancer_stage.isin(["4S", "IV"])]
print(counts_4S_IV)
counts_4S_IV_df = pd.DataFrame(counts_4S_IV.X, index=counts_4S_IV.obs_names, columns=counts_4S_IV.var_names)
counts_4S_IV_df

View of AnnData object with n_obs × n_vars = 67071 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,RGS5,MATR3
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGTGAGA.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGAGGCATGTG.1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGAGTGTACCT.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6,0
T44_AAACCTGCAATGCCAT.1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,0
T44_AAACCTGCAATGGATA.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTTGCGCTCGGATGTT.1,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,5,0
T40_TTTGGTTAGTGGAGTC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
T40_TTTGGTTCATAGTAAG.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
counts_I_III = rdata[rdata.obs.discrete_cancer_stage.isin(["I", "III"])]
print(counts_I_III)
counts_I_III_df = pd.DataFrame(counts_I_III.X, index=counts_I_III.obs_names, columns=counts_I_III.var_names)
counts_I_III_df

View of AnnData object with n_obs × n_vars = 47490 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,RGS5,MATR3
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGGTTGC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,0
T44_AAACGGGAGCCACGCT.1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,12,0
T44_AAACGGGTCCACTGGG.1,0,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,8,0
T44_AAAGATGAGAGGGCTT.1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
T44_AAAGATGAGGATGCGT.1,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,12,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TGGGCGTCAGGTCGTC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
T40_TTAACTCTCACCGGGT.1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,16,0
T40_TTGTAGGCAGCGAACA.1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,5,0


In [18]:
counts_I_IV = rdata[rdata.obs.discrete_cancer_stage.isin(["I", "IV"])]
print(counts_I_IV)
counts_I_IV_df = pd.DataFrame(counts_I_IV.X, index=counts_I_IV.obs_names, columns=counts_I_IV.var_names)
counts_I_IV_df

View of AnnData object with n_obs × n_vars = 70846 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,RGS5,MATR3
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGGTTGC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,0
T44_AAACCTGAGGCATGTG.1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGCAATGCCAT.1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,0
T44_AAACCTGGTTGTACAC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
T44_AAACCTGTCCGTAGTA.1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTTGGTTAGCAATATG.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
T40_TTTGGTTAGTGGAGTC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
T40_TTTGGTTCATAGTAAG.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
counts_III_IV = rdata[rdata.obs.discrete_cancer_stage.isin(["III", "IV"])]
print(counts_III_IV)
counts_III_IV_df = pd.DataFrame(counts_III_IV.X, index=counts_III_IV.obs_names, columns=counts_III_IV.var_names)
counts_III_IV_df

View of AnnData object with n_obs × n_vars = 74896 × 12850
    obs: 'sample_source', 'sample_ID', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,RGS5,MATR3
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGGCATGTG.1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
T44_AAACCTGCAATGCCAT.1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,0
T44_AAACCTGGTTGTACAC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
T44_AAACCTGTCCGTAGTA.1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
T44_AAACGGGAGCCACGCT.1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,12,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTTGCGCTCGGATGTT.1,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,5,0
T40_TTTGGTTAGTGGAGTC.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
T40_TTTGGTTCATAGTAAG.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
metadata_4S_III = metadata[metadata.discrete_cancer_stage.isin(["4S", "III"])]
metadata_4S_III

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGAGTGAGA.1,4S
T44_AAACCTGAGTGTACCT.1,4S
T44_AAACCTGCAATGGATA.1,4S
T44_AAACCTGCACTGTGTA.1,4S
T44_AAACCTGGTAATCACC.1,4S
...,...
T40_TTCTTAGCAGTCTTCC.1,4S
T40_TTCTTAGTCTCTAGGA.1,4S
T40_TTGTAGGCAGCGAACA.1,III


In [25]:
metadata_4S_IV = metadata[metadata.discrete_cancer_stage.isin(["4S", "IV"])]
metadata_4S_IV

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGAGTGAGA.1,4S
T44_AAACCTGAGGCATGTG.1,IV
T44_AAACCTGAGTGTACCT.1,4S
T44_AAACCTGCAATGCCAT.1,IV
T44_AAACCTGCAATGGATA.1,4S
...,...
T40_TTTGCGCTCGGATGTT.1,IV
T40_TTTGGTTAGTGGAGTC.1,IV
T40_TTTGGTTCATAGTAAG.1,IV


In [26]:
metadata_I_III = metadata[metadata.discrete_cancer_stage.isin(["I", "III"])]
metadata_I_III

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGAGGTTGC.1,I
T44_AAACGGGAGCCACGCT.1,III
T44_AAACGGGTCCACTGGG.1,I
T44_AAAGATGAGAGGGCTT.1,I
T44_AAAGATGAGGATGCGT.1,I
...,...
T40_TGGGCGTCAGGTCGTC.1,I
T40_TTAACTCTCACCGGGT.1,III
T40_TTGTAGGCAGCGAACA.1,III


In [27]:
metadata_I_IV = metadata[metadata.discrete_cancer_stage.isin(["I", "IV"])]
metadata_I_IV

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGAGGTTGC.1,I
T44_AAACCTGAGGCATGTG.1,IV
T44_AAACCTGCAATGCCAT.1,IV
T44_AAACCTGGTTGTACAC.1,IV
T44_AAACCTGTCCGTAGTA.1,IV
...,...
T40_TTTGGTTAGCAATATG.1,I
T40_TTTGGTTAGTGGAGTC.1,IV
T40_TTTGGTTCATAGTAAG.1,IV


In [28]:
metadata_III_IV = metadata[metadata.discrete_cancer_stage.isin(["III", "IV"])]
metadata_III_IV

,discrete_cancer_stage
Cell_barcode,
T44_AAACCTGAGGCATGTG.1,IV
T44_AAACCTGCAATGCCAT.1,IV
T44_AAACCTGGTTGTACAC.1,IV
T44_AAACCTGTCCGTAGTA.1,IV
T44_AAACGGGAGCCACGCT.1,III
...,...
T40_TTTGCGCTCGGATGTT.1,IV
T40_TTTGGTTAGTGGAGTC.1,IV
T40_TTTGGTTCATAGTAAG.1,IV


preprocessing

In [13]:
# # remove NA
# samples_to_keep = ~metadata.discrete_cancer_stage.isna()
# counts_df = counts_df.loc[samples_to_keep]
# metadata = metadata.loc[samples_to_keep]

In [14]:
# # remove genes with total counts < 10
# genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
# counts_df = counts_df[genes_to_keep]

In [19]:
# # save the file
# with open("dds.pkl", "wb") as f:
#         pkl.dump(dds, f)